# DiffusionBlocks++ ViT on Colab

Run the repository's ViT training script on a Colab GPU. In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another available GPU) before running the cells. This starter run uses 5 epochs; change `EPOCHS` below for longer training.


In [1]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No GPU detected. Select a GPU runtime in Colab, then rerun.')


PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [2]:
# Reuse the existing checkout if this cell has already run.
%cd /content
!test -d /content/DiffusionBlocksPlusPlus || git clone https://github.com/nhford/DiffusionBlocksPlusPlus.git
%cd /content/DiffusionBlocksPlusPlus


/content
Cloning into 'DiffusionBlocksPlusPlus'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 54 (delta 19), reused 35 (delta 11), pack-reused 6 (from 1)
Receiving objects: 100% (54/54), 605.77 KiB | 10.82 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/DiffusionBlocksPlusPlus


In [3]:
# Colab already provides CUDA-enabled PyTorch. Install the project's other deps;
# flash-attn is omitted because this code does not call it and it is CUDA-build specific.
%pip install 'lightning==2.5.0' 'wandb>=0.22.3' 'transformers==4.52.4' 'datasets==3.5.0' 'huggingface-hub[hf-xet]>=0.31.2' 'hf-transfer>=0.1.9' 'scipy>=1.16.2'


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of huggingface-hub[hf-xet] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of huggingface-hub[hf-xet] to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.3/815.3 kB 11.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 96.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━

In [4]:
# Short starter run. Transformers should use PyTorch only; disabling its optional
# TensorFlow integration avoids unrelated TensorFlow/protobuf conflicts in Colab.
%env USE_TF=0
# W&B runs offline; checkpoints are written under logs/.
EPOCHS = 5
BATCH_SIZE = 128
!python main.py train cifar100 --model_type vit --num_epochs {EPOCHS} --batch_size {BATCH_SIZE} --num_workers 2 --debug


env: USE_TF=0
Seed set to 42
Experiment Name: 2026-10-22T05-20-52-vit
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
README.md: 9.98kB [00:00, 33.5MB/s]
cifar100/test-00000-of-00001.parquet: 100% 23.8M/23.8M [00:01<00:00, 20.4MB/s]
cifar100/train-00000-of-00001.parquet: 100% 119M/119M [00:01<00:00, 92.0MB/s]
Setting num_proc from 4 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Generating train split: 100% 50000/50000 [00:00<00:00, 100559.34 examples/s]
Setting num_proc from 4 back to 1 for the test split to disable multiprocessing as it only contains one shard.
Generating test split: 100% 10000/10000 [00:00<00:00, 114822.78 examples/s]
wandb: WARNING The anonymous setting has no effect and will be remo

In [7]:
import wandb
wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: ERROR Invalid API key: API key must have 40+ characters, has 36.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently lo

True

In [9]:
import os
print(os.getcwd())

/content/DiffusionBlocksPlusPlus


In [20]:
!python main.py test cifar100 --model_type vit \
     --ckpt_path "/content/DiffusionBlocksPlusPlus/logs/2026-10-22T05-20-52-vit/last.ckpt"\
         --num_epochs {EPOCHS} --batch_size {BATCH_SIZE} --num_workers 2 \
            # --debug

Seed set to 42
Experiment Name: 2026-10-22T05-20-52-vit
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: changxig (changxig-carnegie-mellon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run 3e495067ed9e464082e002b4a9690f5f (0.4s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in logs/2026-10-22T05-20-52-vit/wandb/run-20261021_210918-3e495067ed9e464082e002b4a9690f5f
wandb: Run `wandb offline` to turn off syncing.
wandb: Sy

In [28]:
!python main.py test cifar100 --model_type vit \
     --ckpt_path "/content/DiffusionBlocksPlusPlus/logs/2026-10-22T05-20-52-vit/last.ckpt"\
         --num_epochs {EPOCHS} --batch_size {BATCH_SIZE} --num_workers 2 \
            --ode "heun"\
            # --debug

Seed set to 42
Experiment Name: 2026-10-22T05-20-52-vit
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: changxig (changxig-carnegie-mellon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in logs/2026-10-22T05-20-52-vit/wandb/run-20261021_214127-1e1b56aabd6e4be4898006929824316b
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run 2026-10-22T05-20-52-vit
wandb: ⭐️ View project at http

In [11]:
# List the saved checkpoint files. Download or copy them to Drive before the runtime ends.
!find logs -name '*.ckpt' -type f

logs/2026-10-22T05-20-52-vit/last.ckpt
logs/2026-10-22T05-20-52-vit/epoch=4-step=1950.ckpt


In [16]:
!grep -n -A 10 "logger = WandbLogger" /content/DiffusionBlocksPlusPlus/main.py

37:    logger = WandbLogger(
38-        project=f"dblocks-{args.data_name}",
39-        name=nowname,
40-        version=nowname,
41-        offline=args.debug,
42-        save_dir=logdir,
43-        # group=f"{args.data_name}",
44-    )
45-    trainer = L.Trainer(
46-        max_epochs=args.num_epochs
47-        if args.model_type != "dblock"


In [19]:
from pathlib import Path

path = Path("/content/DiffusionBlocksPlusPlus/main.py")
source = path.read_text()

if "version=nowname" in source:
    source = "import uuid\n" + source.replace(
        "version=nowname",
        "version=uuid.uuid4().hex",
    )
    path.write_text(source)
    print("Updated W&B run ID generation")
else:
    print("Expected text not found; inspect main.py before changing it")

Updated W&B run ID generation


In [18]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()  # Select main.py from your Mac

if "main.py" in uploaded:
    destination = Path("/content/DiffusionBlocksPlusPlus/main.py")
    destination.write_bytes(uploaded["main.py"])
    print(f"Updated {destination}")

KeyboardInterrupt: 

In [27]:
import base64
import json
from pathlib import Path

payload = r'''{"main.py": "aW1wb3J0IG9zCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgZGF0ZXRpbWUKaW1wb3J0IGpzb24KaW1wb3J0IHRvcmNoCmltcG9ydCBsaWdodG5pbmcgYXMgTApmcm9tIGxpZ2h0bmluZy5weXRvcmNoLmxvZ2dlcnMgaW1wb3J0IFdhbmRiTG9nZ2VyCmZyb20gbGlnaHRuaW5nLnB5dG9yY2guY2FsbGJhY2tzIGltcG9ydCBNb2RlbENoZWNrcG9pbnQsIExlYXJuaW5nUmF0ZU1vbml0b3IKZnJvbSBsaWdodG5pbmcucHl0b3JjaC5zdHJhdGVnaWVzIGltcG9ydCBERFBTdHJhdGVneSwgRGVlcFNwZWVkU3RyYXRlZ3kKCmZyb20gZGF0YSBpbXBvcnQgbG9hZF9kYXRhCmZyb20gbW9kZWwgaW1wb3J0IGxvYWRfbW9kZWwKCnRvcmNoLnNldF9mbG9hdDMyX21hdG11bF9wcmVjaXNpb24oImhpZ2giKQp0b3JjaC5iYWNrZW5kcy5jdWRhLm1hdG11bC5hbGxvd190ZjMyID0gVHJ1ZQp0b3JjaC5iYWNrZW5kcy5jdWRubi5hbGxvd190ZjMyID0gVHJ1ZQoKCmRlZiBtYWluKGFyZ3MpOgogICAgTC5zZWVkX2V2ZXJ5dGhpbmcoYXJncy5zZWVkKQoKICAgIGRhdGEgPSBsb2FkX2RhdGEoYXJncykKICAgIGFyZ3MuaW1hZ2Vfc2l6ZSA9IGRhdGEuaW1hZ2Vfc2l6ZQogICAgYXJncy5udW1fbGFiZWxzID0gZGF0YS5udW1fbGFiZWxzCiAgICBtb2RlbCA9IGxvYWRfbW9kZWwoYXJncykKICAgIGlmIGFyZ3MuY2twdF9wYXRoIGlzIG5vdCBOb25lOgogICAgICAgIG5vd25hbWUgPSBvcy5wYXRoLmJhc2VuYW1lKG9zLnBhdGguZGlybmFtZShhcmdzLmNrcHRfcGF0aCkpCiAgICBlbHNlOgogICAgICAgIG5vdyA9IGRhdGV0aW1lLmRhdGV0aW1lLm5vdygKICAgICAgICAgICAgdHo9ZGF0ZXRpbWUudGltZXpvbmUoZGF0ZXRpbWUudGltZWRlbHRhKGhvdXJzPTkpLCBuYW1lPSJKU1QiKQogICAgICAgICkuc3RyZnRpbWUoIiVZLSVtLSVkVCVILSVNLSVTIikKICAgICAgICBub3duYW1lID0gbm93ICsgZiIte2FyZ3MubW9kZWxfdHlwZX0iICsgYXJncy5wb3N0Zml4CiAgICAgICAgaWYgbm93bmFtZS5zdGFydHN3aXRoKCJfIik6CiAgICAgICAgICAgIG5vd25hbWUgPSBub3duYW1lWzE6XQogICAgcHJpbnQoIkV4cGVyaW1lbnQgTmFtZToiLCBub3duYW1lKQogICAgbG9nZGlyID0gb3MucGF0aC5qb2luKCJsb2dzIiwgbm93bmFtZSkKICAgIGxvZ2dlciA9IFdhbmRiTG9nZ2VyKAogICAgICAgIHByb2plY3Q9ZiJkYmxvY2tzLXthcmdzLmRhdGFfbmFtZX0iLAogICAgICAgIG5hbWU9bm93bmFtZSwKICAgICAgICB2ZXJzaW9uPV9faW1wb3J0X18oInV1aWQiKS51dWlkNCgpLmhleCwKICAgICAgICBvZmZsaW5lPWFyZ3MuZGVidWcsCiAgICAgICAgc2F2ZV9kaXI9bG9nZGlyLAogICAgICAgICMgZ3JvdXA9ZiJ7YXJncy5kYXRhX25hbWV9IiwKICAgICkKICAgIHRyYWluZXIgPSBMLlRyYWluZXIoCiAgICAgICAgbWF4X2Vwb2Nocz1hcmdzLm51bV9lcG9jaHMKICAgICAgICBpZiBhcmdzLm1vZGVsX3R5cGUgIT0gImRibG9jayIKICAgICAgICBlbHNlIGFyZ3MubnVtX2Vwb2NocwogICAgICAgICogYXJncy5udW1fYmxvY2tzLCAgIyB0byBhbGlnbiB0b3RhbCBudW1iZXIgb2YgaXRlcmF0aW9ucyBhY3Jvc3MgdGhlIGVudGlyZSBuZXR3b3JrIGJlY2F1c2Ugb25lIHN0ZXAgY29ycmVzcG9uZHMgdG8gb25lIGJsb2NrCiAgICAgICAgY2hlY2tfdmFsX2V2ZXJ5X25fZXBvY2g9YXJncy5zYXZlX2V2ZXJ5X25fZXBvY2hzLAogICAgICAgIGNhbGxiYWNrcz1bCiAgICAgICAgICAgIE1vZGVsQ2hlY2twb2ludCgKICAgICAgICAgICAgICAgIGRpcnBhdGg9bG9nZGlyLAogICAgICAgICAgICAgICAgbW9uaXRvcj0idmFsL2FjYyIgaWYgZGF0YS52YWxfa2V5IGlzIG5vdCBOb25lIGVsc2UgTm9uZSwKICAgICAgICAgICAgICAgIG1vZGU9Im1heCIsCiAgICAgICAgICAgICAgICBzYXZlX3RvcF9rPWFyZ3Muc2F2ZV90b3BfaywKICAgICAgICAgICAgICAgIHNhdmVfb25fdHJhaW5fZXBvY2hfZW5kPVRydWUsCiAgICAgICAgICAgICAgICBldmVyeV9uX2Vwb2Nocz1hcmdzLnNhdmVfZXZlcnlfbl9lcG9jaHMKICAgICAgICAgICAgICAgIGlmIGRhdGEudmFsX2tleSBpcyBOb25lCiAgICAgICAgICAgICAgICBlbHNlIE5vbmUsCiAgICAgICAgICAgICAgICBzYXZlX2xhc3Q9VHJ1ZSwKICAgICAgICAgICAgKSwKICAgICAgICAgICAgTGVhcm5pbmdSYXRlTW9uaXRvcihsb2dnaW5nX2ludGVydmFsPSJzdGVwIiksCiAgICAgICAgXSwKICAgICAgICBhY2N1bXVsYXRlX2dyYWRfYmF0Y2hlcz1hcmdzLmFjY3VtdWxhdGVfZ3JhZF9iYXRjaGVzLAogICAgICAgIGdyYWRpZW50X2NsaXBfdmFsPTEuMCwKICAgICAgICBzdHJhdGVneT1ERFBTdHJhdGVneShmaW5kX3VudXNlZF9wYXJhbWV0ZXJzPWFyZ3MubW9kZWxfdHlwZSA9PSAiZGJsb2NrIikKICAgICAgICBpZiBhcmdzLmRldmljZXMgPiAxCiAgICAgICAgZWxzZSAiYXV0byIsCiAgICAgICAgZGV2aWNlcz1hcmdzLmRldmljZXMsCiAgICAgICAgbG9nZ2VyPWxvZ2dlciwKICAgICAgICBudW1fc2FuaXR5X3ZhbF9zdGVwcz0wLAogICAgICAgICMgcHJlY2lzaW9uPSJiZjE2LW1peGVkIiwKICAgICkKICAgIGlmIGFyZ3Muc3RhZ2UgPT0gInRyYWluIjoKICAgICAgICB0cmFpbmVyLmZpdChtb2RlbCwgZGF0YSwgY2twdF9wYXRoPWFyZ3MuY2twdF9wYXRoKQogICAgICAgIGlmIGRhdGEudGVzdF9rZXkgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHRyYWluZXIudGVzdChtb2RlbCwgZGF0YS50ZXN0X2RhdGFsb2FkZXIoKSwgY2twdF9wYXRoPSJiZXN0IikKICAgIGVsc2U6CiAgICAgICAgYXNzZXJ0IGFyZ3MuY2twdF9wYXRoIGlzIG5vdCBOb25lCiAgICAgICAgbW9kZWwudGVzdF9rd2FyZ3MgPSB7Im9kZSI6IGFyZ3Mub2RlfQogICAgICAgIHRyYWluZXIudGVzdChtb2RlbCwgZGF0YSwgY2twdF9wYXRoPWFyZ3MuY2twdF9wYXRoKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCJzdGFnZSIsIHR5cGU9c3RyLCBkZWZhdWx0PSJ0cmFpbiIsIGNob2ljZXM9WyJ0cmFpbiIsICJ0ZXN0Il0pCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCJkYXRhX25hbWUiLCB0eXBlPXN0ciwgZGVmYXVsdD0iY2lmYXIxMDAiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgKICAgICAgICAiLS1tb2RlbF90eXBlIiwgdHlwZT1zdHIsIGRlZmF1bHQ9InZpdCIsIGNob2ljZXM9WyJ2aXQiLCAiZGJsb2NrIl0KICAgICkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGV2aWNlcyIsIHR5cGU9aW50LCBkZWZhdWx0PTEpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW51bV9lcG9jaHMiLCB0eXBlPWludCwgZGVmYXVsdD01MDApCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWJhdGNoX3NpemUiLCB0eXBlPWludCwgZGVmYXVsdD0xMjgpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWFkZF9yYW5kX2F1ZyIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWV2YWxfYmF0Y2hfc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PU5vbmUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNhdmVfZXZlcnlfbl9lcG9jaHMiLCB0eXBlPWludCwgZGVmYXVsdD01KQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1hY2N1bXVsYXRlX2dyYWRfYmF0Y2hlcyIsIHR5cGU9aW50LCBkZWZhdWx0PTEpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWdyYWRpZW50X2NoZWNrcG9pbnRpbmciLCBhY3Rpb249InN0b3JlX3RydWUiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1zY2hlZHVsZXJfdHlwZSIsIHR5cGU9c3RyLCBkZWZhdWx0PSJjb25zdGFudF93aXRoX3dhcm11cCIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KAogICAgICAgICItLXNjaGVkdWxlcl9zcGVjaWZpY19rd2FyZ3MiLAogICAgICAgIHR5cGU9anNvbi5sb2FkcywKICAgICAgICBkZWZhdWx0PU5vbmUsCiAgICAgICAgaGVscD0ic3BlY2lmaWMga3dhcmdzIGZvciB0aGUgc2NoZWR1bGVyIiwKICAgICkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbHIiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAuMDAxKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS13ZWlnaHRfZGVjYXkiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAuMDEpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW9wdGltaXplciIsIHR5cGU9c3RyLCBkZWZhdWx0PSJhZGFtdyIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW51bV93YXJtdXBfc3RlcHMiLCB0eXBlPWludCwgZGVmYXVsdD0wKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kZWVwc3BlZWQiLCBhY3Rpb249InN0b3JlX3RydWUiLCBoZWxwPSJ1c2UgZGVlcHNwZWVkIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc2VlZCIsIHR5cGU9aW50LCBkZWZhdWx0PTQyKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1zYXZlX3RvcF9rIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbnVtX3dvcmtlcnMiLCB0eXBlPWludCwgZGVmYXVsdD04KQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1ja3B0X3BhdGgiLCB0eXBlPXN0ciwgZGVmYXVsdD1Ob25lKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kZWJ1ZyIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KAogICAgICAgICItLXBvc3RmaXgiLCB0eXBlPXN0ciwgZGVmYXVsdD0iIiwgaGVscD0icG9zdGZpeCBmb3IgdGhlIGV4cGVyaW1lbnQgbmFtZSIKICAgICkKICAgICMgZGJsb2NrCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW51bV9ibG9ja3MiLCB0eXBlPWludCwgZGVmYXVsdD0zKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1nYW1tYSIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC4wNSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbnVtX2luZmVyZW5jZV9zdGVwcyIsIHR5cGU9aW50LCBkZWZhdWx0PU5vbmUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWNmZ19zY2FsZSIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC4wKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1jbGFzc19kcm9wb3V0X3Byb2IiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAuMCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb2RlIiwgdHlwZT1zdHIsIGRlZmF1bHQ9ImV1bGVyIikKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCiAgICBtYWluKGFyZ3MpCg==", "model.py": "aW1wb3J0IHJhbmRvbQppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBzY2lweS5zdGF0cyBpbXBvcnQgbm9ybQppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgppbXBvcnQgbGlnaHRuaW5nIGFzIEwKaW1wb3J0IHRvcmNobWV0cmljcwpmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgZ2V0X3NjaGVkdWxlcgoKZnJvbSB2aXQgaW1wb3J0IGxvYWRfdml0CmZyb20gZGJsb2NrX21vZHVsZXMgaW1wb3J0IGdldF9ibG9ja19zaWdtYXMsIGdldF9kaXNjcmV0ZV9zaWdtYXMKCgpkZWYgbG9hZF9tb2RlbChhcmdzKToKICAgIGlmIGFyZ3MubW9kZWxfdHlwZSA9PSAidml0IjoKICAgICAgICByZXR1cm4gVmlUTW9kZWwoYXJncykKICAgIGVsaWYgYXJncy5tb2RlbF90eXBlID09ICJkYmxvY2siOgogICAgICAgIHJldHVybiBWaVREQmxvY2tNb2RlbChhcmdzKQogICAgZWxzZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiSW52YWxpZCBtb2RlbCB0eXBlOiB7YXJncy5tb2RlbF90eXBlfSIpCgoKY2xhc3MgVmlUTW9kZWwoTC5MaWdodG5pbmdNb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGFyZ3MpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuYXJncyA9IGFyZ3MKICAgICAgICBzZWxmLmltYWdlX3NpemUgPSBhcmdzLmltYWdlX3NpemUKICAgICAgICBzZWxmLm51bV9sYWJlbHMgPSBhcmdzLm51bV9sYWJlbHMKICAgICAgICBzZWxmLnZhbGlkX21ldHJpY3MgPSB0b3JjaG1ldHJpY3MuTWV0cmljQ29sbGVjdGlvbigKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgImFjYyI6IHRvcmNobWV0cmljcy5BY2N1cmFjeSgKICAgICAgICAgICAgICAgICAgICB0YXNrPSJtdWx0aWNsYXNzIiwgbnVtX2NsYXNzZXM9c2VsZi5udW1fbGFiZWxzCiAgICAgICAgICAgICAgICApLAogICAgICAgICAgICAgICAgImYxIjogdG9yY2htZXRyaWNzLkYxU2NvcmUoCiAgICAgICAgICAgICAgICAgICAgdGFzaz0ibXVsdGljbGFzcyIsIG51bV9jbGFzc2VzPXNlbGYubnVtX2xhYmVscwogICAgICAgICAgICAgICAgKSwKICAgICAgICAgICAgfSwKICAgICAgICAgICAgcHJlZml4PSJ2YWwvIiwKICAgICAgICApCiAgICAgICAgc2VsZi50ZXN0X21ldHJpY3MgPSBzZWxmLnZhbGlkX21ldHJpY3MuY2xvbmUocHJlZml4PSJ0ZXN0LyIpCiAgICAgICAgc2VsZi5zYXZlX2h5cGVycGFyYW1ldGVycyhhcmdzKQoKICAgICAgICAjIGFkZCBkaWZmdXNpb24gc3RlcCBtZXRob2QKICAgICAgICBzZWxmLnRlc3Rfa3dhcmdzID0ge30KCiAgICBkZWYgY29uZmlndXJlX21vZGVsKHNlbGYpOgogICAgICAgIHNlbGYubW9kZWwgPSBsb2FkX3ZpdChpbWFnZV9zaXplPXNlbGYuaW1hZ2Vfc2l6ZSwgbnVtX2xhYmVscz1zZWxmLm51bV9sYWJlbHMpCiAgICAgICAgcHJpbnQoc2VsZi5tb2RlbCkKICAgICAgICBpZiBzZWxmLmFyZ3MuZ3JhZGllbnRfY2hlY2twb2ludGluZzoKICAgICAgICAgICAgc2VsZi5tb2RlbC5ncmFkaWVudF9jaGVja3BvaW50aW5nX2VuYWJsZSgpCgogICAgZGVmIGNvbmZpZ3VyZV9vcHRpbWl6ZXJzKHNlbGYpOgogICAgICAgIG9wdGltaXplciA9IHRvcmNoLm9wdGltLkFkYW1XKAogICAgICAgICAgICBzZWxmLm1vZGVsLnBhcmFtZXRlcnMoKSwKICAgICAgICAgICAgbHI9c2VsZi5hcmdzLmxyLAogICAgICAgICAgICB3ZWlnaHRfZGVjYXk9c2VsZi5hcmdzLndlaWdodF9kZWNheSwKICAgICAgICApCiAgICAgICAgc2NoZWR1bGVyID0gZ2V0X3NjaGVkdWxlcigKICAgICAgICAgICAgbmFtZT1zZWxmLmFyZ3Muc2NoZWR1bGVyX3R5cGUsCiAgICAgICAgICAgIG9wdGltaXplcj1vcHRpbWl6ZXIsCiAgICAgICAgICAgIG51bV93YXJtdXBfc3RlcHM9c2VsZi5hcmdzLm51bV93YXJtdXBfc3RlcHMsCiAgICAgICAgICAgIG51bV90cmFpbmluZ19zdGVwcz1zZWxmLnRyYWluZXIuZXN0aW1hdGVkX3N0ZXBwaW5nX2JhdGNoZXMsCiAgICAgICAgICAgIHNjaGVkdWxlcl9zcGVjaWZpY19rd2FyZ3M9c2VsZi5hcmdzLnNjaGVkdWxlcl9zcGVjaWZpY19rd2FyZ3MsCiAgICAgICAgKQogICAgICAgIHJldHVybiBbb3B0aW1pemVyXSwgW3sic2NoZWR1bGVyIjogc2NoZWR1bGVyLCAiaW50ZXJ2YWwiOiAic3RlcCJ9XQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsICoqa3dhcmdzKToKICAgICAgICByZXR1cm4gc2VsZi5tb2RlbCgqKmt3YXJncykubG9naXRzCgogICAgZGVmIHNoYXJlZF9zdGVwKHNlbGYsIGJhdGNoLCBzdGVwPSJ0cmFpbiIsIHJldHVybl9tZXRyaWNzPUZhbHNlLCAqKmt3YXJncyk6CiAgICAgICAgcGl4ZWxfdmFsdWVzID0gYmF0Y2hbInBpeGVsX3ZhbHVlcyJdCiAgICAgICAgbGFiZWxzID0gYmF0Y2hbImxhYmVscyJdCiAgICAgICAgbG9naXRzID0gc2VsZihwaXhlbF92YWx1ZXM9cGl4ZWxfdmFsdWVzLCAqKmt3YXJncykKICAgICAgICBpZiByZXR1cm5fbWV0cmljczoKICAgICAgICAgICAgaWYgc3RlcCA9PSAidmFsIjoKICAgICAgICAgICAgICAgIHJldHVybiBzZWxmLnZhbGlkX21ldHJpY3MoCiAgICAgICAgICAgICAgICAgICAgbG9naXRzLnZpZXcoLTEsIHNlbGYubnVtX2xhYmVscyksIGxhYmVscy52aWV3KC0xKQogICAgICAgICAgICAgICAgKQogICAgICAgICAgICBlbGlmIHN0ZXAgPT0gInRlc3QiOgogICAgICAgICAgICAgICAgcmV0dXJuIHNlbGYudGVzdF9tZXRyaWNzKAogICAgICAgICAgICAgICAgICAgIGxvZ2l0cy52aWV3KC0xLCBzZWxmLm51bV9sYWJlbHMpLCBsYWJlbHMudmlldygtMSkKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHJhaXNlIE5vdEltcGxlbWVudGVkRXJyb3IoZiJTdGVwIHtzdGVwfSBpcyBub3Qgc3VwcG9ydGVkIikKCiAgICAgICAgbG9zcyA9IEYuY3Jvc3NfZW50cm9weShsb2dpdHMudmlldygtMSwgc2VsZi5udW1fbGFiZWxzKSwgbGFiZWxzLnZpZXcoLTEpKQogICAgICAgIGxvc3NfZGljdCA9IHtmIntzdGVwfS9sb3NzIjogbG9zc30KICAgICAgICByZXR1cm4gbG9zcywgbG9zc19kaWN0CgogICAgZGVmIGdldF9tb2RlbF9rd2FyZ3Moc2VsZiwgYmF0Y2gpOgogICAgICAgIHJldHVybiB7fQoKICAgIGRlZiB0cmFpbmluZ19zdGVwKHNlbGYsIGJhdGNoLCBiYXRjaF9pZHgpOgogICAgICAgIGJhdGNoX3NpemUgPSBiYXRjaFsicGl4ZWxfdmFsdWVzIl0uc2hhcGVbMF0KICAgICAgICBtb2RlbF9rd2FyZ3MgPSBzZWxmLmdldF9tb2RlbF9rd2FyZ3MoYmF0Y2gpCiAgICAgICAgbG9zcywgbG9zc19kaWN0ID0gc2VsZi5zaGFyZWRfc3RlcChiYXRjaCwgc3RlcD0idHJhaW4iLCAqKm1vZGVsX2t3YXJncykKICAgICAgICBzZWxmLmxvZ19kaWN0KGxvc3NfZGljdCwgYmF0Y2hfc2l6ZT1iYXRjaF9zaXplLCBwcm9nX2Jhcj1UcnVlKQogICAgICAgIHJldHVybiBsb3NzCgogICAgZGVmIHZhbGlkYXRpb25fc3RlcChzZWxmLCBiYXRjaCwgYmF0Y2hfaWR4KToKICAgICAgICBiYXRjaF9zaXplID0gYmF0Y2hbInBpeGVsX3ZhbHVlcyJdLnNoYXBlWzBdCiAgICAgICAgbW9kZWxfa3dhcmdzID0gc2VsZi5nZXRfbW9kZWxfa3dhcmdzKGJhdGNoKQogICAgICAgIHJlcyA9IHNlbGYuc2hhcmVkX3N0ZXAoYmF0Y2gsIHN0ZXA9InZhbCIsIHJldHVybl9tZXRyaWNzPVRydWUsICoqbW9kZWxfa3dhcmdzKQogICAgICAgIHNlbGYubG9nX2RpY3QocmVzLCBiYXRjaF9zaXplPWJhdGNoX3NpemUsIHByb2dfYmFyPVRydWUpCgogICAgZGVmIHRlc3Rfc3RlcChzZWxmLCBiYXRjaCwgYmF0Y2hfaWR4KToKICAgICAgICBiYXRjaF9zaXplID0gYmF0Y2hbInBpeGVsX3ZhbHVlcyJdLnNoYXBlWzBdCiAgICAgICAgbW9kZWxfa3dhcmdzID0gc2VsZi5nZXRfbW9kZWxfa3dhcmdzKGJhdGNoKQogICAgICAgIGNvbWJpbmVkX2t3YXJncyA9IHsqKm1vZGVsX2t3YXJncywgKipzZWxmLnRlc3Rfa3dhcmdzfQogICAgICAgIHJlcyA9IHNlbGYuc2hhcmVkX3N0ZXAoYmF0Y2gsIHN0ZXA9InRlc3QiLCByZXR1cm5fbWV0cmljcz1UcnVlLCAqKmNvbWJpbmVkX2t3YXJncykKICAgICAgICBzZWxmLmxvZ19kaWN0KHJlcywgYmF0Y2hfc2l6ZT1iYXRjaF9zaXplLCBwcm9nX2Jhcj1UcnVlKQoKCmNsYXNzIFZpVERCbG9ja01vZGVsKFZpVE1vZGVsKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBhcmdzKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKGFyZ3MpCiAgICAgICAgc2VsZi5nYW1tYSA9IGFyZ3MuZ2FtbWEKICAgICAgICBzZWxmLnNpZ21hX2RhdGEgPSAwLjUKICAgICAgICBzZWxmLmNmZ19zY2FsZSA9IGFyZ3MuY2ZnX3NjYWxlCiAgICAgICAgc2VsZi5jbGFzc19kcm9wb3V0X3Byb2IgPSAoCiAgICAgICAgICAgIGFyZ3MuY2xhc3NfZHJvcG91dF9wcm9iIGlmIHNlbGYuY2ZnX3NjYWxlID4gMC4wIGVsc2UgMC4wCiAgICAgICAgKQogICAgICAgIHNlbGYubnVtX2luZmVyZW5jZV9zdGVwcyA9IHNlbGYuYXJncy5udW1faW5mZXJlbmNlX3N0ZXBzIG9yIHNlbGYuYXJncy5udW1fYmxvY2tzCiAgICAgICAgc2VsZi5ibG9ja19zaWdtYXMgPSBnZXRfYmxvY2tfc2lnbWFzKG51bV9sYXllcnM9c2VsZi5hcmdzLm51bV9ibG9ja3MpCiAgICAgICAgc2VsZi5sYXllcl9hc3NpZ25tZW50ID0gTm9uZQogICAgICAgIHNlbGYucmVnaXN0ZXJfYnVmZmVyKAogICAgICAgICAgICAic2lnbWFzIiwKICAgICAgICAgICAgZ2V0X2Rpc2NyZXRlX3NpZ21hcyhudW1fc3RlcHM9c2VsZi5udW1faW5mZXJlbmNlX3N0ZXBzLCBkYmxvY2s9VHJ1ZSkudG8oCiAgICAgICAgICAgICAgICBzZWxmLmRldmljZQogICAgICAgICAgICApLAogICAgICAgICkKICAgICAgICBzZWxmLnNhdmVfaHlwZXJwYXJhbWV0ZXJzKAogICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAiZ2FtbWEiOiBzZWxmLmdhbW1hLAogICAgICAgICAgICAgICAgIm51bV9pbmZlcmVuY2Vfc3RlcHMiOiBzZWxmLm51bV9pbmZlcmVuY2Vfc3RlcHMsCiAgICAgICAgICAgICAgICAiY2ZnX3NjYWxlIjogc2VsZi5jZmdfc2NhbGUsCiAgICAgICAgICAgICAgICAiY2xhc3NfZHJvcG91dF9wcm9iIjogc2VsZi5jbGFzc19kcm9wb3V0X3Byb2IsCiAgICAgICAgICAgIH0sCiAgICAgICAgKQoKICAgIGRlZiBjb25maWd1cmVfbW9kZWwoc2VsZik6CiAgICAgICAgc2VsZi5tb2RlbCA9IGxvYWRfdml0KAogICAgICAgICAgICBpbWFnZV9zaXplPXNlbGYuaW1hZ2Vfc2l6ZSwgbnVtX2xhYmVscz1zZWxmLm51bV9sYWJlbHMsIGlzX2RibG9jaz1UcnVlCiAgICAgICAgKQogICAgICAgIHByaW50KHNlbGYubW9kZWwpCgogICAgZGVmIG5vcm1hbGl6ZV9lbWJlZGRpbmdzKHNlbGYsIHg6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgIHJldHVybiBGLm5vcm1hbGl6ZSh4LCBwPTIsIGRpbT0tMSkKCiAgICBkZWYgZ2V0X2VtYmVkcygKICAgICAgICBzZWxmLCBpbnB1dF9pZHM6IHRvcmNoLlRlbnNvciwgaXNfaW5wdXQ6IGJvb2wgPSBUcnVlCiAgICApIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICBpZiBpc19pbnB1dDoKICAgICAgICAgICAgZW1iZWRzID0gc2VsZi5tb2RlbC5nZXRfaW5wdXRfZW1iZWRkaW5ncygpKGlucHV0X2lkcykKICAgICAgICBlbHNlOgogICAgICAgICAgICBlbWJlZHMgPSBGLmVtYmVkZGluZygKICAgICAgICAgICAgICAgIGlucHV0X2lkcywgd2VpZ2h0PXNlbGYubW9kZWwuZ2V0X291dHB1dF9lbWJlZGRpbmdzKCkud2VpZ2h0CiAgICAgICAgICAgICkKICAgICAgICByZXR1cm4gc2VsZi5ub3JtYWxpemVfZW1iZWRkaW5ncyhlbWJlZHMpCgogICAgZGVmIGdldF9zaWdtYXMoc2VsZiwgbl9zYW1wbGVzOiBpbnQsIHBfbWVhbjogZmxvYXQgPSAtMS4yLCBwX3N0ZDogZmxvYXQgPSAxLjIpOgogICAgICAgIGJsb2NrX2lkeCA9IHJhbmRvbS5jaG9pY2VzKHJhbmdlKHNlbGYuYXJncy5udW1fYmxvY2tzKSwgaz0xKVswXQogICAgICAgIHNpZ21hX21pbl9ibG9jayA9IHNlbGYuYmxvY2tfc2lnbWFzW2Jsb2NrX2lkeF0KICAgICAgICBzaWdtYV9tYXhfYmxvY2sgPSBzZWxmLmJsb2NrX3NpZ21hc1tibG9ja19pZHggKyAxXQogICAgICAgICMgZXh0ZW5kIHRoZSByYW5nZQogICAgICAgIGlmIHNlbGYuZ2FtbWEgPiAwLjA6CiAgICAgICAgICAgIGxvZ19zaWdtYV9taW4gPSBucC5sb2coc2lnbWFfbWluX2Jsb2NrKQogICAgICAgICAgICBsb2dfc2lnbWFfbWF4ID0gbnAubG9nKHNpZ21hX21heF9ibG9jaykKICAgICAgICAgICAgbG9nX3JhbmdlID0gbG9nX3NpZ21hX21heCAtIGxvZ19zaWdtYV9taW4KICAgICAgICAgICAgc2lnbWFfbWluX2Jsb2NrID0gbnAuZXhwKGxvZ19zaWdtYV9taW4gLSBzZWxmLmdhbW1hICogbG9nX3JhbmdlKQogICAgICAgICAgICBzaWdtYV9tYXhfYmxvY2sgPSBucC5leHAobG9nX3NpZ21hX21heCArIHNlbGYuZ2FtbWEgKiBsb2dfcmFuZ2UpCiAgICAgICAgICAgIHNpZ21hX21pbl9ibG9jayA9IG1heChzaWdtYV9taW5fYmxvY2ssIHNlbGYuYmxvY2tfc2lnbWFzWzBdKQogICAgICAgICAgICBzaWdtYV9tYXhfYmxvY2sgPSBtaW4oc2lnbWFfbWF4X2Jsb2NrLCBzZWxmLmJsb2NrX3NpZ21hc1stMV0pCgogICAgICAgIGNkZl9taW5fYmxvY2sgPSBub3JtLmNkZigobnAubG9nKHNpZ21hX21pbl9ibG9jaykgLSBwX21lYW4pIC8gcF9zdGQpCiAgICAgICAgY2RmX21heF9ibG9jayA9IG5vcm0uY2RmKChucC5sb2coc2lnbWFfbWF4X2Jsb2NrKSAtIHBfbWVhbikgLyBwX3N0ZCkKCiAgICAgICAgcmFuZCA9IG5wLnJhbmRvbS51bmlmb3JtKGNkZl9taW5fYmxvY2ssIGNkZl9tYXhfYmxvY2ssIG5fc2FtcGxlcykKICAgICAgICBzaWdtYSA9IG5wLmV4cChwX21lYW4gKyBwX3N0ZCAqIG5vcm0ucHBmKHJhbmQpKQogICAgICAgIHNpZ21hID0gdG9yY2guZnJvbV9udW1weShzaWdtYSkKICAgICAgICByZXR1cm4gc2lnbWEKCiAgICBkZWYgZ2V0X3dlaWdodHMoc2VsZiwgc2lnbWFzKToKICAgICAgICByZXR1cm4gKHNpZ21hcyoqMiArIHNlbGYuc2lnbWFfZGF0YSoqMikgLyAoc2lnbWFzICogc2VsZi5zaWdtYV9kYXRhKSAqKiAyCgogICAgZGVmIGVzdGltYXRlX3RhcmdldF9sYXllcihzZWxmLCBzaWdtYTogdG9yY2guVGVuc29yKSAtPiBpbnQ6CiAgICAgICAgYmxvY2tfc2lnbWFzID0gdG9yY2gudGVuc29yKHNlbGYuYmxvY2tfc2lnbWFzLCBkZXZpY2U9c2lnbWEuZGV2aWNlKQogICAgICAgIGJsb2NrX2lkeCA9IHRvcmNoLmJ1Y2tldGl6ZShzaWdtYSwgYmxvY2tfc2lnbWFzLCByaWdodD1UcnVlKSAtIDEKICAgICAgICBibG9ja19pZHggPSAoc2VsZi5hcmdzLm51bV9ibG9ja3MgLSAxKSAtIGJsb2NrX2lkeAogICAgICAgIGJsb2NrX2lkeCA9IHRvcmNoLmNsYW1wKGJsb2NrX2lkeCwgMCwgc2VsZi5hcmdzLm51bV9ibG9ja3MgLSAxKS5sb25nKCkKICAgICAgICB2YWx1ZXMsIGNvdW50cyA9IGJsb2NrX2lkeC51bmlxdWUocmV0dXJuX2NvdW50cz1UcnVlKQogICAgICAgIHJldHVybiB2YWx1ZXNbY291bnRzLmFyZ21heCgpXS5pdGVtKCkKCiAgICBkZWYgZGVub2lzZShzZWxmLCB4LCB6dCwgc2lnbWEsIGJsb2NrX2lkeD1Ob25lKToKICAgICAgICBpZiBibG9ja19pZHggaXMgTm9uZToKICAgICAgICAgICAgYmxvY2tfaWR4ID0gc2VsZi5lc3RpbWF0ZV90YXJnZXRfbGF5ZXIoc2lnbWEpCiAgICAgICAgaWYgc2VsZi5jbGFzc19kcm9wb3V0X3Byb2IgPiAwLjAgYW5kIHNlbGYudHJhaW5pbmc6CiAgICAgICAgICAgIGRyb3BfeCA9IHRvcmNoLnJhbmQoeC5zaGFwZVswXSwgZGV2aWNlPXguZGV2aWNlKSA8IHNlbGYuY2xhc3NfZHJvcG91dF9wcm9iCiAgICAgICAgICAgIHVuY29uZF94ID0gdG9yY2guemVyb3NfbGlrZSh4KQogICAgICAgICAgICB4ID0gdG9yY2gud2hlcmUoZHJvcF94WzosIE5vbmUsIE5vbmUsIE5vbmVdLCB1bmNvbmRfeCwgeCkKICAgICAgICBlbGlmIG5vdCBzZWxmLnRyYWluaW5nIGFuZCBzZWxmLmNmZ19zY2FsZSA+IDAuMDoKICAgICAgICAgICAgdW5jb25kX3ggPSB0b3JjaC56ZXJvc19saWtlKHgpCiAgICAgICAgICAgIHggPSB0b3JjaC5jYXQoW3VuY29uZF94LCB4XSkKICAgICAgICAgICAgenQgPSB0b3JjaC5jYXQoW3p0XSAqIDIpCiAgICAgICAgICAgIHNpZ21hID0gdG9yY2guY2F0KFtzaWdtYV0gKiAyKQoKICAgICAgICBjX3NraXAgPSBzZWxmLnNpZ21hX2RhdGEqKjIgLyAoc2lnbWEqKjIgKyBzZWxmLnNpZ21hX2RhdGEqKjIpCiAgICAgICAgY19vdXQgPSBzaWdtYSAqIHNlbGYuc2lnbWFfZGF0YSAvIChzaWdtYSoqMiArIHNlbGYuc2lnbWFfZGF0YSoqMikgKiogMC41CiAgICAgICAgY19pbiA9IDEgLyAoc2lnbWEqKjIgKyBzZWxmLnNpZ21hX2RhdGEqKjIpICoqIDAuNQogICAgICAgIGNfbm9pc2UgPSAwLjI1ICogc2lnbWEubG9nKCkKCiAgICAgICAgaWYgc2VsZi5sYXllcl9hc3NpZ25tZW50IGlzIE5vbmU6CiAgICAgICAgICAgIHNwbGl0X3NpemUgPSBzZWxmLm1vZGVsLmNvbmZpZy5udW1faGlkZGVuX2xheWVycyAvLyBzZWxmLmFyZ3MubnVtX2Jsb2NrcwogICAgICAgICAgICBzZWxmLmxheWVyX2Fzc2lnbm1lbnQgPSBbCiAgICAgICAgICAgICAgICBsaXN0KHJhbmdlKGkgKiBzcGxpdF9zaXplLCAoaSArIDEpICogc3BsaXRfc2l6ZSkpCiAgICAgICAgICAgICAgICBmb3IgaSBpbiByYW5nZShzZWxmLmFyZ3MubnVtX2Jsb2NrcykKICAgICAgICAgICAgXQogICAgICAgIG91dHB1dHMgPSBzZWxmLm1vZGVsLmZvcndhcmRfYmxvY2soCiAgICAgICAgICAgIGxheWVyX2luZGljZXM9c2VsZi5sYXllcl9hc3NpZ25tZW50W2Jsb2NrX2lkeF0sCiAgICAgICAgICAgIHBpeGVsX3ZhbHVlcz14LAogICAgICAgICAgICBub2lzeV9lbWJlZHM9enQgKiBjX2luWzosIE5vbmVdLAogICAgICAgICAgICB0aW1lc3RlcHM9Y19ub2lzZSwKICAgICAgICApCiAgICAgICAgaGlkZGVuX3N0YXRlcyA9IG91dHB1dHMubGFzdF9oaWRkZW5fc3RhdGUKICAgICAgICBjb25kaXRpb25pbmcgPSBvdXRwdXRzLmNvbmRpdGlvbmluZwogICAgICAgIG1vZGVsX291dCA9IGhpZGRlbl9zdGF0ZXMgKiBjX291dFs6LCBOb25lXSArIHp0ICogY19za2lwWzosIE5vbmVdCiAgICAgICAgbG9naXRzID0gc2VsZi5tb2RlbC5mb3J3YXJkX291dHB1dF9lbWJlZGRpbmdzKAogICAgICAgICAgICBtb2RlbF9vdXQudW5zcXVlZXplKDEpLCBjb25kaXRpb25pbmcKICAgICAgICApCiAgICAgICAgaWYgbm90IHNlbGYudHJhaW5pbmcgYW5kIHNlbGYuY2ZnX3NjYWxlID4gMC4wOgogICAgICAgICAgICBsb2dpdHNfdW5jb25kLCBsb2dpdHNfY29uZCA9IGxvZ2l0cy5jaHVuaygyKQogICAgICAgICAgICBsb2dpdHMgPSBsb2dpdHNfdW5jb25kICsgc2VsZi5jZmdfc2NhbGUgKiAobG9naXRzX2NvbmQgLSBsb2dpdHNfdW5jb25kKQogICAgICAgIHJldHVybiBsb2dpdHMKCiAgICBkZWYgc2hhcmVkX3N0ZXAoc2VsZiwgYmF0Y2gsIHN0ZXA9InRyYWluIiwgcmV0dXJuX21ldHJpY3M9RmFsc2UsICoqa3dhcmdzKToKICAgICAgICBwaXhlbF92YWx1ZXMgPSBiYXRjaFsicGl4ZWxfdmFsdWVzIl0KICAgICAgICBsYWJlbHMgPSBiYXRjaFsibGFiZWxzIl0KICAgICAgICBvZGUgPSBrd2FyZ3MuZ2V0KCJvZGUiLCAiZXVsZXIiKQoKICAgICAgICBpZiByZXR1cm5fbWV0cmljczoKICAgICAgICAgICAgbG9naXRzID0gc2VsZi5kaWZmdXNpb25fc3RlcChwaXhlbF92YWx1ZXMsb2RlPW9kZSkKICAgICAgICAgICAgaWYgc3RlcCA9PSAidmFsIjoKICAgICAgICAgICAgICAgIHJldHVybiBzZWxmLnZhbGlkX21ldHJpY3MoCiAgICAgICAgICAgICAgICAgICAgbG9naXRzLnZpZXcoLTEsIHNlbGYubnVtX2xhYmVscyksIGxhYmVscy52aWV3KC0xKQogICAgICAgICAgICAgICAgKQogICAgICAgICAgICBlbGlmIHN0ZXAgPT0gInRlc3QiOgogICAgICAgICAgICAgICAgcmV0dXJuIHNlbGYudGVzdF9tZXRyaWNzKAogICAgICAgICAgICAgICAgICAgIGxvZ2l0cy52aWV3KC0xLCBzZWxmLm51bV9sYWJlbHMpLCBsYWJlbHMudmlldygtMSkKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHJhaXNlIE5vdEltcGxlbWVudGVkRXJyb3IoZiJTdGVwIHtzdGVwfSBpcyBub3Qgc3VwcG9ydGVkIikKCiAgICAgICAgeiA9IHNlbGYuZ2V0X2VtYmVkcyhsYWJlbHMsIGlzX2lucHV0PVRydWUpCiAgICAgICAgc2lnbWFzID0gc2VsZi5nZXRfc2lnbWFzKHouc2hhcGVbMF0pCiAgICAgICAgYmxvY2tfaWR4ID0gc2VsZi5lc3RpbWF0ZV90YXJnZXRfbGF5ZXIoc2lnbWFzKQogICAgICAgIHNpZ21hcyA9IHNpZ21hcy50byh6KQogICAgICAgIHp0ID0geiArIHNpZ21hc1s6LCBOb25lXSAqIHRvcmNoLnJhbmRuX2xpa2UoeikKICAgICAgICBsb2dpdHMgPSBzZWxmLmRlbm9pc2UocGl4ZWxfdmFsdWVzLCB6dCwgc2lnbWFzLCBibG9ja19pZHgpCiAgICAgICAgbG9zcyA9IEYuY3Jvc3NfZW50cm9weSgKICAgICAgICAgICAgbG9naXRzLnZpZXcoLTEsIHNlbGYubnVtX2xhYmVscyksIGxhYmVscy52aWV3KC0xKSwgcmVkdWN0aW9uPSJub25lIgogICAgICAgICkKICAgICAgICBjZV9sb3NzID0gbG9zcy5tZWFuKCkKICAgICAgICB3ID0gc2VsZi5nZXRfd2VpZ2h0cyhzaWdtYXMpWzosIE5vbmVdCiAgICAgICAgbG9zcyA9IChsb3NzICogdykubWVhbigpCgogICAgICAgIGxvc3NfZGljdCA9IHsKICAgICAgICAgICAgZiJ7c3RlcH0vbG9zcyI6IGxvc3MsCiAgICAgICAgICAgIGYie3N0ZXB9L2xvc3Nfe2Jsb2NrX2lkeH0iOiBsb3NzLAogICAgICAgICAgICBmIntzdGVwfS9jZV9sb3NzIjogY2VfbG9zcywKICAgICAgICAgICAgZiJ7c3RlcH0vY2VfbG9zc197YmxvY2tfaWR4fSI6IGNlX2xvc3MsCiAgICAgICAgfQogICAgICAgIHJldHVybiBsb3NzLCBsb3NzX2RpY3QKCiAgICBkZWYgZGlmZnVzaW9uX3N0ZXAoc2VsZiwgeCwgb2RlPSdldWxlcicpOgogICAgICAgIGJzeiA9IHguc2hhcGVbMF0KICAgICAgICBoaWRkZW5fc2l6ZSA9IHNlbGYubW9kZWwuY29uZmlnLmhpZGRlbl9zaXplCiAgICAgICAgeiA9IHRvcmNoLnJhbmRuKGJzeiwgaGlkZGVuX3NpemUsIGRldmljZT1zZWxmLmRldmljZSkKICAgICAgICB6ICo9IHRvcmNoLnNxcnQoMS4wICsgc2VsZi5zaWdtYXNbMF0gKiogMi4wKQogICAgICAgIHNfaW4gPSB4Lm5ld19vbmVzKFt4LnNoYXBlWzBdXSkKICAgICAgICBmb3IgaSBpbiByYW5nZShzZWxmLnNpZ21hcy5zaGFwZVswXSAtIDEpOgogICAgICAgICAgICBzaWdtYSA9IHNlbGYuc2lnbWFzW2ldICogc19pbgogICAgICAgICAgICBuZXh0X3NpZ21hID0gc2VsZi5zaWdtYXNbaSArIDFdICogc19pbgogICAgICAgICAgICAjIGRlbm9pc2UgYXQgY3VycmVudCAKICAgICAgICAgICAgbG9naXRzID0gc2VsZi5kZW5vaXNlKHgsIHosIHNpZ21hKQogICAgICAgICAgICBwcm9icyA9IEYuc29mdG1heChsb2dpdHMsIGRpbT0xKQogICAgICAgICAgICBkZW5vaXNlZCA9IEYubGluZWFyKHByb2JzLCBzZWxmLm1vZGVsLmdldF9pbnB1dF9lbWJlZGRpbmdzKCkud2VpZ2h0LnQoKSkKICAgICAgICAgICAgIyB0byBkCiAgICAgICAgICAgIGQgPSAoeiAtIGRlbm9pc2VkKSAvIHNpZ21hWzosIE5vbmVdCiAgICAgICAgICAgIGR0ID0gbmV4dF9zaWdtYSAtIHNpZ21hCiAgICAgICAgICAgICMgZXVsZXIgc3RlcAogICAgICAgICAgICBldWxlcl9zdGVwID0geiArIGR0WzosIE5vbmVdICogZAoKICAgICAgICAgICAgaWYgb2RlID09ICdldWxlcic6CiAgICAgICAgICAgICAgICB6ID0gZXVsZXJfc3RlcAogICAgICAgICAgICBlbGlmIG9kZSA9PSAnaGV1bic6CiAgICAgICAgICAgICAgICAjIFRPRE86IHByZWRpY3QgdGhlIG5leHQgImRlbm9pc2VkIiBhdCB0aGUgcG9zaXRpb24gb2YgdGhlIGV1bGVyIHN0ZXAKICAgICAgICAgICAgICAgIGxvZ2l0cyA9IHNlbGYuZGVub2lzZSh4LCBldWxlcl9zdGVwLCBuZXh0X3NpZ21hKSAjIFRPRE86IHNob3VsZCBzaWdtYSBiZSBuZXh0IHNpZ21hPwogICAgICAgICAgICAgICAgcHJvYnMgPSBGLnNvZnRtYXgobG9naXRzLCBkaW09MSkKICAgICAgICAgICAgICAgIGRlbm9pc2VkMSA9IEYubGluZWFyKHByb2JzLCBzZWxmLm1vZGVsLmdldF9pbnB1dF9lbWJlZGRpbmdzKCkud2VpZ2h0LnQoKSkKCiAgICAgICAgICAgICAgICAjIFRPRE86IGNvbXB1dGUgdGhlIGF2ZXJhZ2UgImRlbm9pc2luZyBzdGVwIgogICAgICAgICAgICAgICAgZDIgPSAoZXVsZXJfc3RlcCAtIGRlbm9pc2VkMSkgLyBuZXh0X3NpZ21hWzosIE5vbmVdCiAgICAgICAgICAgICAgICBoZXVuX3N0ZXAgPSB6ICsgZHRbOiwgTm9uZV0gKiAoZCArIGQyKSAvIDIKICAgICAgICAgICAgICAgIAogICAgICAgICAgICAgICAgeiA9IGhldW5fc3RlcAogICAgICAgICAgICAKICAgICAgICBtaW5fc2lnbWEgPSBzZWxmLnNpZ21hc1stMV0uaXRlbSgpCiAgICAgICAgc2lnbWFzID0gdG9yY2guZnVsbCgoeC5zaGFwZVswXSwpLCBtaW5fc2lnbWEsIGRldmljZT1zZWxmLmRldmljZSkKICAgICAgICBsb2dpdHMgPSBzZWxmLmRlbm9pc2UoeCwgeiwgc2lnbWFzKQogICAgICAgIHJldHVybiBsb2dpdHMK"}
'''

root = Path("/content/DiffusionBlocksPlusPlus")
for name, encoded in json.loads(payload).items():
    (root / name).write_bytes(base64.b64decode(encoded))
    print(f"Updated {name}")

Updated main.py
Updated model.py
